# Train the mineral detector on Colab GPU

**Before running:** `Runtime -> Change runtime type -> Hardware accelerator: T4 GPU -> Save`.

Then run the cells top to bottom. Total time on a T4: ~30-45 min for 100 epochs.
The result `best.pt` downloads to your computer at the end; copy it to the Pi as
`models/ore_best.pt`.

## 1. Confirm the GPU is active
If this errors or shows no GPU, you didn't switch the runtime type above.

In [ ]:
!nvidia-smi

## 2. Install dependencies

In [ ]:
!pip install -q ultralytics roboflow

## 3. Get the dataset

**Option A (recommended) - download via Roboflow API.** Paste your own key
(app.roboflow.com -> Settings -> API Keys). This is the same dataset as your
local one: workspace `mineraldetectionyolo`, project `mineral-c42yg`, version 8.

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="YOUR_ROBOFLOW_KEY")  # <-- paste your key
project = rf.workspace("mineraldetectionyolo").project("mineral-c42yg")
dataset = project.version(8).download("yolov11")
DATA = dataset.location + "/data.yaml"
print("data.yaml at:", DATA)

**Option B (no key) - upload the zip you already have.** Skip cell A and run
this instead; it opens a file picker. Choose `mineral ---.v8i.yolov11.zip`
from your Downloads.

In [ ]:
# from google.colab import files
# up = files.upload()                      # pick the .zip
# name = next(iter(up))
# !mkdir -p dataset && unzip -o -q "$name" -d dataset
# DATA = "dataset/data.yaml"
# print("data.yaml at:", DATA)

## 4. Train (100 epochs, imgsz 640)
Colab auto-uses the GPU (`device=0`).

In [ ]:
from ultralytics import YOLO
model = YOLO("yolo11n.pt")
model.train(data=DATA, epochs=100, imgsz=640, device=0)

## 5. Download the trained weights
Then on your machine, copy to the Pi:

```
scp ~/Downloads/best.pt firestingray:~/tokyo_robotics/image_recognition/models/ore_best.pt
```

In [ ]:
from google.colab import files
files.download("runs/detect/train/weights/best.pt")